# Походження даних і вибірки реплікації

P03 отримує та перевіряє Student Performance. Мета — простежувана підготовка, а не перевірка гіпотез.

Використовуємо [протокол v1.0](../docs/replication_protocol.md): лише `Medu >= 1` та `Fedu >= 1`. Зберігаємо G3=0, дублікати й вихідний порядок. Повний валідований набір із освітою0 зберігається окремо для майбутніх перевірок.

Ядро: **Python (parentedu-replication)**. Запуск: **Restart Kernel and Run All Cells**. Перший запуск після checkout потребує мережі; наявний перевірений кеш використовується без повторного завантаження. Рядки учнів не показуємо в цьому технічному звіті.

## 1. Середовище та зафіксований протокол

Імпортуємо встановлений пакет із src. Перевіряємо реальне ядро, пакети й актуальність lock. Немає імпутації, train/test або автоматичного очищення під потрібний N.

In [ ]:
from IPython.display import display

from parentedu.config import load_config
from parentedu.data_workflow import run_data_preparation
from parentedu.environment import check_environment
from parentedu.paths import ROOT

config = load_config()
checks = check_environment()
print(f"Python {checks['python']}; protocol v{config['protocol']['version']}; seed {checks['seed']}")

## 2. Джерело, кеш і перевірка схеми

Повний процес перевіряє п'ять вихідних файлів за [зафіксованими хешами](../docs/data_provenance.json), читає33поля за схемою, готує два шари та перевіряє read-back CSV. Якщо дані не проходять перевірку, метадані отримують status=failed; старі вихідні файли не слід використовувати як актуальний успішний результат.

Час першого отримання raw зберігається окремо від часу виконання. Якщо раніше дані не завантажувалися, вони отримуються з офіційного UCI; нову невідому версію не приймаємо мовчки.

In [ ]:
run = run_data_preparation()
manifest = run["manifest"]
print(f"DOI: {manifest['doi']}; license: {manifest['license']}")
print(f"Verified local cache: {manifest['cache_hit']}")
print(f"Acquired at UTC: {manifest['retrieved_at_utc']}")
print(f"Pinned source files: {len(manifest['files'])}")
print(f"Quality checks: {len(run['quality'])}; statuses: {sorted(run['quality']['status'].unique())}")

## 3. Облік рядків та освіти0

Лічильники Medu0/Fedu0 перетинаються: сумарне виключення — їх об'єднання. G3=0 залишається, якщо освіта обох батьків відповідає фільтру. Службовий record_id означає запис предмета, не ідентичність людини між предметами.

In [ ]:
display(run["samples"]["flow"])
display(run["samples"]["comparison"])
assert run["samples"]["comparison"]["status"].eq("match").all()
print("All replication sample sizes match the frozen references.")

## 4. Артефакти та межі

Повні й реплікаційні CSV та журнал включення — локально в data/processed.
У Git зберігаємо лише код, схему, паспорт, словник, provenance і агреговані таблиці.
Метадані запуску й хеші outputs — у outputs/logs/data_run.json.
[Паспорт](../docs/data_passport.md) пояснює походження й обмеження; [словник](../docs/data_dictionary.csv) відтворюється зі схеми.

In [ ]:
print(f"Run status: {run['metadata']['status']}")
for name, path in run["files"].items():
    print(f"{name}: {path.relative_to(ROOT)}")
print("Student data prepared. No regression, p-values or hypothesis tests have been run.")

## Наступний крок

P04 — описовий аналіз: розподіли, поєднання освіти батьків і ризики малих груп.
Готовність вибірок ще не означає проходження K1: EDA та решта умов попереду.
Невідомий авторський SE і відкрита бібліографічна вимога P02 залишаються в [матриці вимог](../docs/course_requirements.md).